In [0]:
# ============================================================
# Cell 1 - Install dependencies
# ============================================================
%pip install databricks-langchain databricks-agents mlflow langchain langgraph
dbutils.library.restartPython()

In [0]:
# ============================================================
# Cell 2 - Configuration and imports
# ============================================================
import mlflow
import json
from databricks.sdk import WorkspaceClient
from databricks_langchain import ChatDatabricks
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
import pyspark.sql.functions as F

CATALOG = "media"
BRONZE  = "bronze_tmdb"
SILVER  = "silver_tmdb"
GOLD    = "gold_tmdb"

spark.sql(f"USE CATALOG {CATALOG}")
mlflow.langchain.autolog()
w = WorkspaceClient()

print("Ready!")

In [0]:
# ============================================================
# Cell 3 - Define agent tools
# ============================================================
@tool
def list_tables(schema: str = "bronze_tmdb") -> str:
    """Lists all tables in a given schema. Options: bronze_tmdb, silver_tmdb, gold_tmdb"""
    spark.sql(f"USE CATALOG {CATALOG}")
    tables = spark.sql(f"SHOW TABLES IN {schema}").toPandas()
    return tables.to_json(orient="records", indent=2)


@tool
def profile_table(table_name: str) -> str:
    """
    Profiles a Delta table — returns row count, schema, and null counts.
    Use just the table name e.g. 'raw_movies'. 
    Set the schema first using USE SCHEMA if needed.
    """
    spark.sql(f"USE CATALOG {CATALOG}")
    df = spark.table(table_name)

    row_count  = df.count()
    schema     = [(f.name, str(f.dataType)) for f in df.schema.fields]
    null_counts = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ]).collect()[0].asDict()

    return json.dumps({
        "table":       table_name,
        "row_count":   row_count,
        "schema":      schema,
        "null_counts": null_counts
    }, indent=2)


@tool
def sample_payload(table_name: str, n: int = 3) -> str:
    """
    Returns n sample raw_payload records from a Bronze table.
    Use just the table name e.g. 'raw_movies'
    """
    spark.sql(f"USE CATALOG {CATALOG}")
    rows    = spark.table(table_name).select("raw_payload").limit(n).collect()
    samples = [json.loads(r["raw_payload"]) for r in rows]
    return json.dumps(samples, indent=2)


@tool
def query_table(sql: str) -> str:
    """
    Runs any SQL query against the media catalog.
    Always use fully qualified names: media.<schema>.<table>
    e.g. SELECT * FROM media.gold_tmdb.dim_title LIMIT 5
    """
    try:
        result = spark.sql(sql).limit(50).toPandas()
        return result.to_json(orient="records", indent=2)
    except Exception as e:
        return f"Error: {str(e)}"


tools = [list_tables, profile_table, sample_payload, query_table]
print(f"Tools ready: {[t.name for t in tools]}")

In [0]:
# ============================================================
# Cell 4 - Create the agent
# ============================================================
system_prompt = """You are a senior data engineer helping design and build 
a dimensional model for a streaming media analytics platform on Databricks.

You have access to Delta tables in the media catalog with three layers:
- media.bronze_tmdb  — raw JSON from TMDB API
- media.silver_tmdb  — cleaned, typed tables
- media.gold_tmdb    — star schema dimensional model

When asked to analyze data or write code:
1. Always inspect the actual data first using your tools
2. Base all recommendations on what you actually see
3. Be specific — suggest real column names, data types, transformations
4. Think about the business questions the model should answer
5. Flag any data quality issues you find

The target star schema grain is one row per title per platform:
- fact_title_performance
- dim_title, dim_platform, dim_genre, dim_date, dim_language, dim_person
"""

llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",
    temperature=0.1
)

agent = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt
)

print("Agent ready!")

In [0]:
# ============================================================
# Cell 5 - Conversational chat with memory
# ============================================================
conversation_history = []

def chat(message: str):
    """Send a message to the agent maintaining conversation history."""
    conversation_history.append({"role": "user", "content": message})
    print(f"You: {message}\n")

    response = agent.invoke({"messages": conversation_history})

    for msg in response["messages"]:
        if msg.type == "tool":
            print(f"  [Tool: {msg.name}] -> {msg.content[:200]}...\n")
        elif msg.type == "ai" and msg.content:
            answer = msg.content

    conversation_history.append({"role": "assistant", "content": answer})
    print(f"Agent: {answer}\n")
    return answer

def reset_chat():
    """Clear conversation history to start fresh."""
    global conversation_history
    conversation_history = []
    print("Conversation reset!")

In [0]:
# ============================================================
# Cell 6 - Start chatting!
# ============================================================
reset_chat()
chat("List the tables in bronze_tmdb, silver_tmdb and gold_tmdb and give me a summary of what we have")

In [0]:
chat("Which genres have the highest average rating? Query the gold layer to find out.")

In [0]:
chat("What are the top 10 most popular titles right now and are they movies or TV shows?")

In [0]:
chat("Which languages produce the most content in our dataset?")